# Module 7: Deploy

> Self-directed module, ~25 min. **Optional** — covers shipping the agent via LangSmith Deployments. Skip if you don't have deployment permissions or are using LangSmith strictly for observability and evaluations.

Ship the client research agent to **LangSmith Deployments** using the `langgraph` CLI. A deployed agent is reachable through 30+ endpoints — REST, MCP, A2A, Agent Protocol, HITL, persistent memory, and the Studio UI — without any additional plumbing.

**Steps:**

1. Inspect the deployable layout under `agents/deployable_agents/client_research/`
2. Review `langgraph.json` and the agent's `AGENTS.md`
3. Validate the configuration locally
4. Deploy to LangSmith
5. Exercise the deployed agent in Studio

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print(f"LANGSMITH_API_KEY: {'set' if os.environ.get('LANGSMITH_API_KEY') else 'MISSING'}")
print(f"TAVILY_API_KEY:    {'set' if os.environ.get('TAVILY_API_KEY') else 'MISSING'}")

# Report whichever model-provider key is configured.
for var in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "AZURE_OPENAI_API_KEY", "AWS_ACCESS_KEY_ID"]:
    if os.environ.get(var):
        print(f"{var}: set")


## 1. Project structure

A deployable LangGraph project is a directory containing `langgraph.json` at the root that references one or more graph objects. `agents/deployable_agents/client_research/` holds the agent code, identity, and skills; `langgraph.json` (at the repo root) registers it.


In [ ]:
agent_dir = str(project_root / "agents" / "deployable_agents" / "client_research")
print("agents/deployable_agents/client_research/")
print("---")

for root, dirs, files in os.walk(agent_dir):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


## 2. Configuration

`langgraph.json` declares the deploy configuration: dependencies, graphs, and the env file. The LangGraph CLI reads it for both `langgraph dev` (local server) and `langgraph deploy` (push to LangSmith).

Two graphs are registered. `client_research` is the primary deployable for this module; `base_research_agent` is included as a second example for inspection.

`AGENTS.md` is the agent's identity — loaded into the prompt via the `memory=` argument in `create_deep_agent()` and editable by the agent itself at runtime.


In [ ]:
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


The agent itself lives at `agents/deployable_agents/client_research/agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"client_research": "./agents/deployable_agents/client_research/agent.py:agent"`.


## 3. Local development

Three CLI commands, all run from the repo root (where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks dependencies)
langgraph validate

# Run locally with hot reload + Studio UI (long-running — open in a separate terminal)
langgraph dev --port 2024

# Deploy to LangSmith
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in the browser — useful for stepping through tool calls before deploying. It's a long-running server, so run it in a separate terminal rather than from this notebook.

Reference: [LangGraph CLI](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).


## 4. Validate the configuration

`langgraph validate` imports each graph in `langgraph.json` and verifies dependencies resolve — without building Docker or uploading anything. Use it to catch configuration and import errors before deploying.


In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


If `langgraph validate` fails, the most common cause is a missing dependency in `pyproject.toml` or an import error in `agents/deployable_agents/client_research/agent.py`. Fix locally before running `langgraph deploy` — the deploy build will fail the same way, just slower.


## 5. Deploy to LangSmith

`langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.

> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status


In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
!cd "{project_root}" && langgraph deploy --name langsmith-poc-client-research --no-input


Deploys take a few minutes to provision. While waiting:

- Watch progress in the **Deployments** tab in LangSmith.
- Open `langgraph dev` in a separate terminal to iterate locally against the same `langgraph.json`.
- Tail logs once provisioning starts: `langgraph deploy logs <deployment-id>`.


### Deployment endpoints

When `langgraph deploy` finishes, the CLI prints the deployment's base URL. Two endpoints worth bookmarking:

- **Studio / home:** the base URL itself — linked from the **Deployments** tab in LangSmith. Opens the deployed graph in Studio.

<img src="../images/deployment_home.png" alt="LangSmith deployment home page" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

- **OpenAPI / Swagger UI:** append `/docs` to the base URL — interactive API reference showing every endpoint the deployment exposes (`/runs`, `/threads`, `/store`, etc.).

<img src="../images/deployment_docs.png" alt="OpenAPI Swagger UI for the deployed agent" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

The same URLs are available via `langgraph deploy list` if you lose the original output.


## 6. What you get with LangSmith Deployments

Once deployed, the agent is reachable through 30+ endpoints — built once, exposed everywhere:

| Capability | What it enables |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call this one as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads via the platform's Store |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger the agent on a schedule |

Both registered graphs (`client_research` and `base_research_agent`) are reachable individually through these endpoints once the deployment is live.


## 7. Test the deployed agent in Studio

With the deployment provisioned, exercise the agent through the **Studio UI** — linked from the deployment's page in LangSmith, or available standalone at `https://smith.langchain.com/studio` against any registered graph. Studio is a chat-style interface that streams responses, surfaces HITL interrupts inline, and persists each conversation as a thread that lands in the trace project.

Running a representative batch of queries here validates that the deployed agent behaves the same way the local one did. The queries also feed additional trace volume into your LangSmith project, useful for Module 3's failure-mode discovery features.

<img src="../images/Studio_testing.png" alt="LangSmith Studio mid-conversation with the deployed client_research agent" style="width: auto; max-height: 380px; border-radius: 8px;">

### Test query checklist

Open Studio against the `client_research` graph and run each query in order. Each one targets a distinct trace shape; together they exercise every code path the agent has.

**Profile lookup only (single tool, fastest)**

1. `What does Smith Family Office hold?`
2. `Pull up Acme Pension Fund and tell me about their risk profile.`
3. `What does TechCorp Treasury hold?`

**Research only (research subagent, no client lookup)**

4. `What's the latest news on Apple (AAPL)? Search once.`
5. `Summarize Microsoft's most recent quarterly earnings.`
6. `Research recent news on the energy sector. Search at most once.`

**Multi-step (profile + research, longer trajectories)**

7. `Prep me for my meeting with Smith Family Office. Cover their top three holdings briefly.`
8. `Look up Acme Pension Fund, then research one major holding.`
9. `Write a portfolio update for TechCorp Treasury and save it to /tcorp_update.md.`

**Edge cases (likely to surface failure modes)**

10. `Look up the client 'globex-corp' and tell me about their holdings.`
11. `What was Smith Family Office's exact return last quarter?`
12. `How will AAPL stock perform next month?`


### What to watch for

While running through the checklist:

- **Trajectory** — profile-only queries should show one `get_client_profile` call in the trace tree. Research-only queries should show one `task` (subagent delegation) and one or more `web_search` calls inside the subagent. Multi-step queries should chain `get_client_profile` → `task` → `write_file`.
- **Unknown client handling** — query 10 should report the client is not in the system and list known IDs, not fabricate data.
- **Speculative refusal** — query 12 asks for a prediction the agent can't responsibly make. The compliance rules in the agent's middleware (Module 1, §1.5) should result in a hedged or declined response, not a confident forecast.

Optional: repeat the checklist a second time to roughly double the trace volume. Module 7's Insights Agent benefits from a larger corpus, though the warm-up and trigger prompts from earlier modules will have already produced enough traces for it to work with.


## Recap

| File | Purpose |
|---|---|
| `langgraph.json` | LangGraph deploy config (dependencies, graphs, env file) |
| `agents/deployable_agents/client_research/agent.py` | The deployable `agent` object (built with `create_deep_agent`) |
| `AGENTS.md` | Agent identity and instructions (loaded via `memory=`) |
| `skills/` | On-demand capability modules — `client-brief`, `portfolio-update` (loaded via `skills=`) |
| `deepagents.toml` | Per-agent name and model config |
| `pyproject.toml` | Python dependencies for the deploy image |

**Operational follow-ups:**
- `langgraph deploy list` — list deployments
- `langgraph deploy logs <id>` — tail logs
- `langgraph deploy revisions list <id>` — revision history
- Re-run `langgraph deploy` to push a new revision; the CLI finds the existing deployment by name.

**Next:** Return to `03_finding_failure_modes.ipynb` to analyze the expanded trace corpus from Studio testing using Chat, Insights, and Engine.